# Module 1 · Lesson 03: Your First Anthropic (Claude) API Call

Now that you know OpenAI's API, let's learn **Anthropic's Claude** — a major competitor.
Understanding *both* APIs makes you a versatile AI engineer.

## What you will learn
1. How to call Claude using the `anthropic` SDK
2. **Key differences** between OpenAI and Anthropic APIs
3. System prompts in Claude (separate parameter!)
4. Multi-turn conversations with Claude
5. Streaming with Claude

---
### Prerequisites
- `ANTHROPIC_API_KEY` set in `.env`
- `pip install anthropic`

Get an API KEY: https://platform.claude.com/settings/keys

Models and pricing overview: https://platform.claude.com/docs/en/about-claude/models/overview

In [2]:
# ── Setup ──
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown, clear_output

load_dotenv(Path.cwd().parent / ".env")

import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY
MODEL = "claude-sonnet-4-6"

if client:
    print(f"Anthropic client ready — using {MODEL}")

Anthropic client ready — using claude-sonnet-4-6


---
## 1. Basic Completion

Claude uses `client.messages.create()` instead of `client.chat.completions.create()`.

| OpenAI | Anthropic |
|--------|----------|
| `chat.completions.create()` | `messages.create()` |
| `response.choices[0].message.content` | `response.content[0].text` |
| `max_tokens` optional | `max_tokens` **required** |
| System = message in list | System = **separate parameter** |

In [ ]:
# ── Example 1: Basic Claude completion ──
response = client.messages.create(
    model=MODEL,
    max_tokens=1000, # Required in Anthropic!           
    messages=[
        {
            "role": "user", 
            "content": "What is Python? Answer in one sentence."
        },
    ]
)

answer = response.content[0].text    # Note: .content[0].text, not .choices[0].message.content
display(Markdown(f"**Response:** {answer}"))

# Token usage
print(f"=> Tokens - Input: {response.usage.input_tokens}, Output: {response.usage.output_tokens}")

---
## 2. System Prompt — A Key Difference!

In Claude, the system prompt is a **separate parameter**, not a message in the list:

```python
# OpenAI style (system is a message)
messages=[
    {"role": "system", "content": "You are a tutor"},
    {"role": "user", "content": "..."}
]

# Anthropic style (system is a parameter)
client.messages.create(
    system="You are a tutor",     # ← separate!
    messages=[{"role": "user", "content": "..."}]
)
```

In [ ]:
# ── Example 2: System prompt in Claude ────────────────
response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    system="You are a helpful programming tutor. Explain concepts simply using analogies.",
    messages=[
        {"role": "user", "content": "What is recursion?"}
    ]
)

display(Markdown(f"### Claude Tutor\n\n{response.content[0].text}"))

---
## 3. Temperature Comparison

In [ ]:
# ── Example 3: Temperature ────────────────────────────
prompt = "Write a one-sentence story about a robot."

for temp in [0.0, 1.0]:
    response = client.messages.create(
        model=MODEL,
        max_tokens=60,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp
    )
    label = "Deterministic" if temp == 0 else "Creative"
    display(Markdown(f"**Temperature {temp} ({label}):** {response.content[0].text}"))

---
## 4. Multi-Turn Conversation

Claude can also remember — use same method as OpenAI: send full history.

In [ ]:
# ── Example 4: Multi-turn conversation with Claude ──

prompt_intro = "My name is Alice."
prompt_followup = "What is my name?"

conversation = [
    {"role": "user", "content": prompt_intro},
]

# Turn 1 - (1st API call)
response_turn1 = client.messages.create(model=MODEL, max_tokens=50, messages=conversation)
assistant_reply1 = response_turn1.content[0].text
print(f"User:    {prompt_intro}")
print(f"Claude:  {assistant_reply1}\n")

# Turn 2 - (2nd API call)
conversation.append({"role": "assistant", "content": assistant_reply1})
conversation.append({"role": "user", "content": prompt_followup})

response_turn2 = client.messages.create(model=MODEL, max_tokens=50, messages=conversation)
assistant_reply2 = response_turn2.content[0].text
print(f"User:    {prompt_followup}")
print(f"Claude:  {assistant_reply2}")

---
## 5. Streaming with Claude

Streaming lets tokens appear one-by-one — essential for responsive UIs.

In [ ]:
# ── Example 5: Streaming ──
print("Streaming response:\n")

with client.messages.stream(
    model=MODEL,
    max_tokens=150,
    messages=[{"role": "user", "content": "Explain what an API is in 3 sentences."}]
) as stream:
    full_text = ""
    for text in stream.text_stream:
        # print(text, end="", flush=True)
        full_text += text
        clear_output(wait=True)
        display(Markdown(full_text))

print(f"\n\nTotal characters: {len(full_text)}")

---
## 6. Claude Model Family (2026)

| Model | API ID | Latency | Best For |
|-------|--------|---------|----------|
| **Claude Opus 4.6** | `claude-opus-4-6` | Moderate | Most intelligent - agents & coding |
| **Claude Sonnet 4.6** | `claude-sonnet-4-6` | Fast | Best combo of speed & intelligence |
| **Claude Haiku 4.5** | `claude-haiku-4-5-20251001` | Fastest | Near-frontier intelligence, low cost |

### Pricing (per 1M tokens)
| Model | Input | Output | Context Window | Max Output |
|-------|------:|-------:|---------------:|-----------:|
| Opus 4.6 | $5.00 | $25.00 | 200K (1M beta) | 128K |
| Sonnet 4.6 | $3.00 | $15.00 | 200K (1M beta) | 64K |
| Haiku 4.5 | $1.00 | $5.00 | 200K | 64K |

---
## 7. API Differences Cheat Sheet

| Feature | OpenAI | Anthropic |
|---------|--------|-----------|
| SDK | `openai` | `anthropic` |
| Client | `OpenAI()` | `Anthropic()` |
| Method | `chat.completions.create()` | `messages.create()` |
| System prompt | Message in list | Separate `system=` param |
| Response text | `response.choices[0].message.content` | `response.content[0].text` |
| Max tokens | Optional (has default) | **Required** |
| Token count | `.usage.prompt_tokens` | `.usage.input_tokens` |
| Streaming | `stream=True` | `messages.stream()` context manager |

---
## Key Takeaways

1. **Same concepts, different syntax** - both APIs use messages, roles, temperature
2. **Choose by task**: Haiku for speed, Sonnet for balance, Opus for quality
3. **System prompt location** is the biggest API difference
4. **Being multi-provider** gives you flexibility and fallback options

---
**Next:** `04_compare_models.ipynb` - Compare OpenAI vs Claude side by side